# Table 4: All-Material GBM Comparison

## Step 1: Run all-material GBM comparison

This step computes Raw-18, Raw-21, global sigma-star, and regularized temperature-dependent sigma-star GBM results for each material and family using 5-fold cross-validation.


In [1]:
"""
All materials: correction-magnitude-regularized sigma*_T-18.
"""

import os
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import KFold

try:
    from numpy.exceptions import RankWarning
except ImportError:
    RankWarning = np.RankWarning
warnings.simplefilter("ignore", RankWarning)


DATA_PATH = os.path.join("..", "Dataset", "Creep_Properties_Dataset.xlsx")

COMP_FEATURES = [
    "C, wt.%", "Si, wt.%", "Mn, wt.%", "P, wt.%", "S, wt.%", "Ni, wt.%",
    "Cr, wt.%", "Mo, wt.%", "Al, wt.%", "N, wt.%", "Ti, wt.%", "Fe, wt.%",
    "Nb, wt.%", "B, wt.%", "Cu, wt.%", "Co, wt.%",
]
GEO_FEATURES   = ["Thickness (mm)", "Cross-Sectional Area", "Aspect Ratio (L/D)"]
BASE18_FEATURES = ["Stress (Mpa)", "Temperature of Creep Test (C)"] + COMP_FEATURES
BASE21_FEATURES = BASE18_FEATURES + GEO_FEATURES

GBM_PARAMS = dict(
    n_estimators=500, max_depth=5, learning_rate=0.03,
    subsample=0.8, min_samples_leaf=3, random_state=42,
)
GBM_PARAMS_SMALL = dict(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.8, min_samples_leaf=3, random_state=42,
)
N_SPLITS = 5
MIN_N    = N_SPLITS
LAMBDAS  = [0.0, 0.0003, 0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0]

RPV_ALLOYS = {"SA533B", "SA508", "A508/A533"}

DATASETS = {
    "SS316H":      {"alloys": ["SS316H"],                              "sigma_params": 3},
    "SS304H":      {"alloys": ["SS304H"],                              "sigma_params": 3},
    "SS316L":      {"alloys": ["SS316L"],                              "sigma_params": 3},
    "SA533B":      {"alloys": ["SA533B"],                              "sigma_params": 2},
    "SS316 Family":{"alloys": ["SS316H","SS316L","SS316LN","SS316FR"], "sigma_params": 3},
    "SS304 Family":{"alloys": ["SS304H","SS304J","SS304L"],            "sigma_params": 3},
    "RPV Family":  {"alloys": ["SA533B","SA508","A508/A533"],          "sigma_params": 2},
    "SS316LN":     {"alloys": ["SS316LN"],                             "sigma_params": 3},
    "SS316FR":     {"alloys": ["SS316FR"],                             "sigma_params": 3},
    "SS304J":      {"alloys": ["SS304J"],                              "sigma_params": 3},
    "SS304L":      {"alloys": ["SS304L"],                              "sigma_params": 3},
    "A709":        {"alloys": ["A709"],                                "sigma_params": 3},
    "IN-617":      {"alloys": ["IN-617"],                              "sigma_params": 3},
    "IN-600":      {"alloys": ["IN-600"],                              "sigma_params": 3},
    "Zr-3":        {"alloys": ["Zr-3"],                                "sigma_params": 3},
    "SA508":       {"alloys": ["SA508"],                               "sigma_params": 2},
    "A508/A533":   {"alloys": ["A508/A533"],                           "sigma_params": 2},
}


# -----------------------------------------------------------------------
# DATA FIXES
# -----------------------------------------------------------------------
def impute_rpv_cross_section(df: pd.DataFrame) -> pd.DataFrame:
    """Cross-Sectional Area for RPV steels.
    """
    df = df.copy()
    rpv_mask = df["Alloy Designation"].isin(RPV_ALLOYS)
    missing   = rpv_mask & df["Cross-Sectional Area"].isna()

    # Cylindrical: compute from diameter
    cyl = missing & (df["Specimen Shape"] == "Cylindrical")
    d   = df.loc[cyl, "Thickness (mm)"]
    df.loc[cyl, "Cross-Sectional Area"] = np.pi * (d / 2) ** 2
    n_cyl = int(cyl.sum())

    # Flat: impute with per-alloy median
    flat = missing & (df["Specimen Shape"] == "Flat")
    rpv_median_overall = df.loc[rpv_mask & df["Cross-Sectional Area"].notna(),
                                "Cross-Sectional Area"].median()
    n_flat = 0
    for alloy in RPV_ALLOYS:
        alloy_flat = flat & (df["Alloy Designation"] == alloy)
        if not alloy_flat.any():
            continue
        alloy_median = df.loc[
            rpv_mask & (df["Alloy Designation"] == alloy) & df["Cross-Sectional Area"].notna(),
            "Cross-Sectional Area"
        ].median()
        fill_val = alloy_median if not np.isnan(alloy_median) else rpv_median_overall
        df.loc[alloy_flat, "Cross-Sectional Area"] = fill_val
        n_flat += int(alloy_flat.sum())

    print(f"  [RPV] imputed Cross-Sectional Area: "
          f"{n_cyl} cylindrical (formula), {n_flat} flat (median)")
    return df


def impute_ss316L_composition(df: pd.DataFrame) -> pd.DataFrame:
    """Composition for SS316L."""
    df   = df.copy()
    mask = df["Alloy Designation"] == "SS316L"
    comp_missing = mask & df[COMP_FEATURES[0]].isna()   # all 16 cols missing together

    if not comp_missing.any():
        return df

    # Compute medians from records that have data
    medians = df.loc[mask & ~comp_missing, COMP_FEATURES].median()
    for col in COMP_FEATURES:
        df.loc[comp_missing, col] = medians[col]

    print(f"  [SS316L] imputed composition for {int(comp_missing.sum())} records "
          f"using SS316L per-column medians")
    return df


# -----------------------------------------------------------------------
# MODEL HELPERS  (unchanged from original)
# -----------------------------------------------------------------------
def metrics(y_true, y_pred):
    return (
        float(r2_score(y_true, y_pred)),
        float(np.sqrt(mean_squared_error(y_true, y_pred))),
        float(np.mean(np.abs(y_true - y_pred) < np.log10(2)) * 100.0),
    )


def proxy(ls_star, y, T):
    total, n = 0.0, 0
    for tv in np.unique(T):
        mk = T == tv
        if int(np.sum(mk)) < 3:
            continue
        if np.std(ls_star[mk]) < 1e-10:
            resid = y[mk] - np.mean(y[mk])
        else:
            try:
                coef  = np.polyfit(ls_star[mk], y[mk], 1)
                resid = y[mk] - np.polyval(coef, ls_star[mk])
            except np.linalg.LinAlgError:
                resid = y[mk] - np.mean(y[mk])
        total += float(np.var(resid)) * int(np.sum(mk))
        n     += int(np.sum(mk))
    return total / max(n, 1)


def apply_global(p, n_params, ls, lb, lA, lLD):
    if n_params == 3:
        out = ls + p[0]*lb + p[1]*lA + p[2]*lLD
    else:
        out = ls + p[0]*lb + p[1]*lLD
    return np.clip(out, -1.0, 4.0)


def apply_tdep(p, n_params, ls, Tn, lb, lA, lLD):
    if n_params == 3:
        out = ls + (p[0]+p[3]*Tn)*lb + (p[1]+p[4]*Tn)*lA + (p[2]+p[5]*Tn)*lLD
    else:
        out = ls + (p[0]+p[2]*Tn)*lb + (p[1]+p[3]*Tn)*lLD
    return np.clip(out, -1.0, 4.0)


def opt_global(n_params, ls, y, T, lb, lA, lLD):
    x0  = [0.0, 0.0, -0.1] if n_params == 3 else [0.0, -0.1]
    res = minimize(
        lambda p: proxy(apply_global(p, n_params, ls, lb, lA, lLD), y, T),
        x0, method="Nelder-Mead",
        options={"maxiter": 3000, "xatol": 1e-5, "fatol": 1e-6},
    )
    return res.x


def opt_tdep_regularized(n_params, ls, y, T, Tn, lb, lA, lLD, p0, lam):
    x0      = [p0[0], p0[1], p0[2], 0.0, 0.0, 0.0] if n_params == 3 else [p0[0], p0[1], 0.0, 0.0]
    maxiter = 5000 if n_params == 3 else 4000

    def objective(p):
        ls_star = apply_tdep(p, n_params, ls, Tn, lb, lA, lLD)
        return proxy(ls_star, y, T) + lam * float(np.mean((ls_star - ls) ** 2))

    return minimize(objective, x0, method="Nelder-Mead",
                    options={"maxiter": maxiter, "xatol": 1e-5, "fatol": 1e-6}).x


def fit_predict(X, y, tr, val, gbm_params):
    model = GradientBoostingRegressor(**gbm_params)
    model.fit(X[tr], y[tr])
    return model.predict(X[val])


def replace_stress(X, ls_star):
    out      = X.copy()
    out[:, 0] = 10.0 ** ls_star
    return out


def add_result(rows, dataset, alloys, n, sigma_params, gbm_kind,
               status, model, pred, y, raw_r2):
    if status != "ok":
        rows.append({"dataset": dataset, "alloys": ",".join(alloys),
                     "n": n, "sigma_params": sigma_params, "gbm": gbm_kind,
                     "status": status, "model": model,
                     "r2": np.nan, "rmse": np.nan,
                     "within_2x_pct": np.nan, "delta_vs_raw18": np.nan})
        return
    r2, rmse, w2 = metrics(y, pred)
    rows.append({"dataset": dataset, "alloys": ",".join(alloys),
                 "n": n, "sigma_params": sigma_params, "gbm": gbm_kind,
                 "status": status, "model": model,
                 "r2": r2, "rmse": rmse, "within_2x_pct": w2,
                 "delta_vs_raw18": r2 - raw_r2})


# -----------------------------------------------------------------------
# MAIN
# -----------------------------------------------------------------------
print("Loading data ...")
df_raw = pd.read_excel(DATA_PATH, sheet_name="Data", header=1)

df_raw = impute_rpv_cross_section(df_raw)
df_raw = impute_ss316L_composition(df_raw)

rows = []

for ds_name, cfg in DATASETS.items():
    print(f"\n{'='*88}\nDataset: {ds_name}")
    alloys   = cfg["alloys"]
    n_params = cfg["sigma_params"]

    mask = (
        df_raw["Alloy Designation"].isin(alloys)
        & (df_raw["Notch"] == "unnotched")
        & df_raw["Creep Rupture Life (h)"].notna()
        & (df_raw["Creep Rupture Life (h)"] > 0)
        & (df_raw["Stress (Mpa)"] > 0)
        & df_raw["Specimen Shape"].isin(["Flat", "Cylindrical"])
    )
    data = df_raw[mask].copy().reset_index(drop=True)
    data["log_tR"]    = np.log10(data["Creep Rupture Life (h)"])
    data["log_sigma"] = np.log10(data["Stress (Mpa)"])

    extra_cols = []
    for alloy in alloys[1:]:
        col = f"is_{alloy.replace('/', '_')}"
        data[col] = (data["Alloy Designation"] == alloy).astype(float)
        extra_cols.append(col)

    feat18 = BASE18_FEATURES + extra_cols
    feat21 = BASE21_FEATURES + extra_cols
    needed = BASE21_FEATURES + extra_cols + ["log_tR", "log_sigma"]
    sub    = data[needed].dropna(subset=BASE21_FEATURES + ["log_tR"]).reset_index(drop=True)
    n      = len(sub)

    if n < N_SPLITS:
        print(f"  skipped: n={n} < {N_SPLITS} required for {N_SPLITS}-fold CV")
        add_result(rows, ds_name, alloys, n, n_params, "skipped",
                   "n_too_small_for_cv", "skipped", None, None, np.nan)
        continue

    y   = sub["log_tR"].values.astype(float)
    ls  = sub["log_sigma"].values.astype(float)
    T   = sub["Temperature of Creep Test (C)"].values.astype(float)
    Tn  = (T - float(T.mean())) / max(float(T.std()), 1e-9)
    b   = sub["Thickness (mm)"].values.astype(float)
    A   = sub["Cross-Sectional Area"].values.astype(float)
    LD  = sub["Aspect Ratio (L/D)"].values.astype(float)
    b0  = float(np.nanmedian(b));  A0 = float(np.nanmedian(A)); LD0 = float(np.nanmedian(LD))
    lb  = np.log10(np.where(b  > 0, b  / b0,  1.0))
    lA  = np.log10(np.where(A  > 0, A  / A0,  1.0))
    lLD = np.log10(np.where(LD > 0, LD / LD0, 1.0))

    X18       = sub[feat18].values.astype(float)
    X21       = sub[feat21].values.astype(float)
    gbm_kind  = "small" if n < 150 else "full"
    gbm_params = GBM_PARAMS_SMALL if n < 150 else GBM_PARAMS
    print(f"  n={n}, sigma* params={n_params}, gbm={gbm_kind}")

    preds = {"Raw-18": np.zeros(n), "Raw-21": np.zeros(n), "Star-global-18": np.zeros(n)}
    for lam in LAMBDAS:
        preds[f"Reg-Tdep-18 lambda={lam:g}"] = np.zeros(n)

    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
    for fold, (tr, val) in enumerate(kf.split(X18), 1):
        print(f"    Fold {fold}/{N_SPLITS}", flush=True)
        preds["Raw-18"][val] = fit_predict(X18, y, tr, val, gbm_params)
        preds["Raw-21"][val] = fit_predict(X21, y, tr, val, gbm_params)

        p_global   = opt_global(n_params, ls[tr], y[tr], T[tr], lb[tr], lA[tr], lLD[tr])
        ls_global  = apply_global(p_global, n_params, ls, lb, lA, lLD)
        preds["Star-global-18"][val] = fit_predict(
            replace_stress(X18, ls_global), y, tr, val, gbm_params)

        for lam in LAMBDAS:
            p       = opt_tdep_regularized(n_params, ls[tr], y[tr], T[tr], Tn[tr],
                                           lb[tr], lA[tr], lLD[tr], p_global, lam)
            ls_star = apply_tdep(p, n_params, ls, Tn, lb, lA, lLD)
            preds[f"Reg-Tdep-18 lambda={lam:g}"][val] = fit_predict(
                replace_stress(X18, ls_star), y, tr, val, gbm_params)

    raw_r2 = metrics(y, preds["Raw-18"])[0]
    for model, pred in preds.items():
        add_result(rows, ds_name, alloys, n, n_params, gbm_kind, "ok",
                   model, pred, y, raw_r2)

    best_reg_name = max(
        [k for k in preds if k.startswith("Reg-Tdep-18")],
        key=lambda k: metrics(y, preds[k])[0],
    )
    print(f"  Raw18={raw_r2:.4f}  best={best_reg_name} "
          f"R2={metrics(y, preds[best_reg_name])[0]:.4f}")

# -----------------------------------------------------------------------
# SUMMARY
# -----------------------------------------------------------------------
long_df = pd.DataFrame(rows)

summary_rows = []
for dataset, grp in long_df.groupby("dataset", sort=False):
    first = grp.iloc[0]
    if first["status"] != "ok":
        summary_rows.append({
            "dataset": dataset, "alloys": first["alloys"],
            "n": first["n"], "sigma_params": first["sigma_params"],
            "gbm": first["gbm"], "status": first["status"],
            "raw18_r2": np.nan, "raw21_r2": np.nan, "global18_r2": np.nan,
            "tdep18_unregularized_r2": np.nan,
            "best_regularized_model": "skipped",
            "best_regularized_r2": np.nan, "best_delta_vs_raw18": np.nan,
            "extra_vs_unregularized": np.nan,
            "best_regularized_rmse": np.nan, "best_regularized_w2x": np.nan,
        })
        continue
    raw     = grp[grp["model"] == "Raw-18"].iloc[0]
    raw21   = grp[grp["model"] == "Raw-21"].iloc[0]
    global18 = grp[grp["model"] == "Star-global-18"].iloc[0]
    tdep0   = grp[grp["model"] == "Reg-Tdep-18 lambda=0"].iloc[0]
    reg_grp = grp[grp["model"].str.startswith("Reg-Tdep-18")]
    best    = reg_grp.loc[reg_grp["r2"].idxmax()]
    summary_rows.append({
        "dataset": dataset, "alloys": first["alloys"],
        "n": first["n"], "sigma_params": first["sigma_params"],
        "gbm": first["gbm"], "status": first["status"],
        "raw18_r2": raw["r2"], "raw21_r2": raw21["r2"], "global18_r2": global18["r2"],
        "tdep18_unregularized_r2": tdep0["r2"],
        "best_regularized_model": best["model"],
        "best_regularized_r2": best["r2"],
        "best_delta_vs_raw18": best["r2"] - raw["r2"],
        "extra_vs_unregularized": best["r2"] - tdep0["r2"],
        "best_regularized_rmse": best["rmse"],
        "best_regularized_w2x": best["within_2x_pct"],
    })

summary = pd.DataFrame(summary_rows)

print("\n" + "="*120)
print(f"{'Dataset':<16} {'n':>5} {'Raw18':>8} {'Raw21':>8} {'Tdep0':>8} "
      f"{'BestReg':>8} {'Delta':>8} {'Status':<10}")
print("-"*120)
for _, r in summary.iterrows():
    if r["status"] != "ok":
        print(f"{r['dataset']:<16} {int(r['n']):>5} {'':>8} {'':>8} {'':>8} "
              f"{'':>8} {'':>8} {r['status']:<10}")
        continue
    print(f"{r['dataset']:<16} {int(r['n']):>5} "
          f"{r['raw18_r2']:>8.4f} {r['raw21_r2']:>8.4f} "
          f"{r['tdep18_unregularized_r2']:>8.4f} "
          f"{r['best_regularized_r2']:>8.4f} {r['best_delta_vs_raw18']:>+8.4f} "
          f"{r['status']:<10}")
print("="*120)


Loading data ...
  [RPV] imputed Cross-Sectional Area: 15 cylindrical (formula), 47 flat (median)
  [SS316L] imputed composition for 15 records using SS316L per-column medians

Dataset: SS316H
  n=1066, sigma* params=3, gbm=full
    Fold 1/5
    Fold 2/5
    Fold 3/5
    Fold 4/5
    Fold 5/5
  Raw18=0.9436  best=Reg-Tdep-18 lambda=0.01 R2=0.9474

Dataset: SS304H
  n=834, sigma* params=3, gbm=full
    Fold 1/5
    Fold 2/5
    Fold 3/5
    Fold 4/5
    Fold 5/5
  Raw18=0.8874  best=Reg-Tdep-18 lambda=1 R2=0.8955

Dataset: SS316L
  n=120, sigma* params=3, gbm=small
    Fold 1/5
    Fold 2/5
    Fold 3/5
    Fold 4/5
    Fold 5/5
  Raw18=0.8351  best=Reg-Tdep-18 lambda=1 R2=0.8429

Dataset: SA533B
  n=66, sigma* params=2, gbm=small
    Fold 1/5
    Fold 2/5
    Fold 3/5
    Fold 4/5
    Fold 5/5
  Raw18=0.2615  best=Reg-Tdep-18 lambda=1 R2=0.3362

Dataset: SS316 Family
  n=1280, sigma* params=3, gbm=full
    Fold 1/5
    Fold 2/5
    Fold 3/5
    Fold 4/5
    Fold 5/5
  Raw18=0.9428  bes

## Final Table 4


In [2]:
COLUMN_MAP = {
    "dataset": "Dataset",
    "n": "n",
    "raw18_r2": "Nominal stress sigma",
    "raw21_r2": "Nominal stress sigma with geometry features",
    "best_regularized_r2": "Regularized temperature-dependent geometry-corrected stress sigma_T*",
    "best_regularized_model": "Best regularized model",
}

PAPER_ROW_ORDER = [
    "SS316 Family", "SS304 Family", "RPV Family",
    "SS316H", "SS304H", "SS316L",
    "SS304J", "SS316FR", "A709", "IN-617", "SS316LN",
]

final_table = summary[list(COLUMN_MAP)].rename(columns=COLUMN_MAP).reset_index(drop=True)
final_table = (
    final_table.set_index("Dataset")
    .loc[PAPER_ROW_ORDER]
    .reset_index()
)
final_table

,Dataset,n,Nominal stress sigma,Nominal stress sigma with geometry features,Regularized temperature-dependent geometry-corrected stress sigma_T*,Best regularized model
0,SS316 Family,1280,0.942837,0.944051,0.942200,Reg-Tdep-18 lambda=1
1,SS304 Family,923,0.901758,0.903011,0.908351,Reg-Tdep-18 lambda=1
2,RPV Family,135,0.589335,0.578986,0.686734,Reg-Tdep-18 lambda=0.003
3,SS316H,1066,0.943602,0.944543,0.947403,Reg-Tdep-18 lambda=0.01
4,SS304H,834,0.887409,0.887251,0.895457,Reg-Tdep-18 lambda=1
5,SS316L,120,0.835130,0.851048,0.842876,Reg-Tdep-18 lambda=1
6,SS304J,69,0.921789,0.922334,0.914036,Reg-Tdep-18 lambda=0.3
7,SS316FR,61,0.878638,0.877110,0.878638,Reg-Tdep-18 lambda=0
8,A709,52,0.362027,0.499534,0.596123,Reg-Tdep-18 lambda=0
9,IN-617,43,0.630664,0.605152,0.640471,Reg-Tdep-18 lambda=0
